# meteo_export — exporta scada_proc y publica en SharePoint

Lee `scada_proc` por planta y publica un CSV en SharePoint **directamente con la API de Microsoft Graph**, usando una app de Azure AD con permisos de escritura. No depende del shortcut SharePoint del lakehouse.

**Destino SharePoint:** site `https://enerlandgroup.sharepoint.com/sites/Pvsyst1`, biblioteca `Documentos compartidos`, subcarpeta configurable (`inbox` por defecto).


In [1]:
# Parámetros
plant_codes  = None
period_start  = None
period_end    = None
resolution    = "native"        # 'native' | '1H'

# --- App registrada en Azure AD: "PVSyst1_EPI" ---
# client_id y tenant_id son identificadores públicos (no son secretos).
client_id  = "58f369e5-0dfd-4435-802b-a90a6786bf22"
tenant_id  = "943fa85b-068d-404a-bacb-5f0baf8d31a8"

# El client_secret SÍ es sensible: vive en Key Vault.
key_vault_url             = "https://apicredentialsenerland.vault.azure.net/"
secret_client_secret_name = "PVSyst1EPISecret"

# --- SharePoint destino ---
sp_site_path  = "enerlandgroup.sharepoint.com:/sites/Pvsyst1"
sp_drive_name = "Documentos"     # nombre real de la biblioteca en el site
sp_inbox_path = "inbox"

# --- Audit local (copia en OneLake, escritura siempre garantizada) ---
local_audit_root = "inbox_audit"

In [2]:
def _norm_plant_codes(v):
    """Normaliza plant_codes a lista (una o varias). None/'' = todas.
    Acepta lista/tupla, JSON array o cadena separada por comas."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


import io
import requests
import pandas as pd
import numpy as np
import notebookutils
from datetime import datetime
from pathlib import Path

LAKEHOUSE_FILES = '/lakehouse/default/Files'
ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read_sheet(sheet, table=None):
    """Lee una TABLA ESTRUCTURADA del Excel de config: 1) por NOMBRE de tabla
    (case-insensitive, en cualquier hoja); 2) si no, la (primera) tabla de la hoja
    indicada; 3) ultimo recurso, la hoja completa. Robusto ante renombrados."""
    local = '/tmp/plants_config_meteo.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    import openpyxl
    wb = openpyxl.load_workbook(local, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(local, sheet_name=sheet)
def get_active_plants():
    df = _read_sheet('01_Plants', 'T01_plants')
    df = df[df['pr_execution'].astype(str).str.upper().isin(['TRUE','VERDADERO','1','SÍ','SI'])]
    _pl = _norm_plant_codes(plant_codes)
    if _pl:
        df = df[df['plant_code'].isin(_pl)]
    return df.to_dict(orient='records')


In [3]:
# ---------------------------------------------------------------------------
# Microsoft Graph helpers
# ---------------------------------------------------------------------------
_GRAPH_BASE = "https://graph.microsoft.com/v1.0"
_TIMEOUT    = 60


def get_graph_token():
    """Client credentials OAuth flow contra la app de Azure AD 'PVSyst1_EPI'.
    client_id y tenant_id están en código (no son secretos);
    el client_secret se lee de Key Vault."""
    client_secret = notebookutils.credentials.getSecret(key_vault_url, secret_client_secret_name)

    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    data = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=data, timeout=_TIMEOUT)
    if not r.ok:
        raise RuntimeError(f"OAuth fallo {r.status_code}: {r.text[:400]}")
    return r.json()['access_token']


def _gget(token, path):
    r = requests.get(f"{_GRAPH_BASE}/{path.lstrip('/')}",
                     headers={'Authorization': f'Bearer {token}'},
                     timeout=_TIMEOUT)
    if not r.ok:
        raise RuntimeError(f"Graph GET {path} -> {r.status_code}: {r.text[:400]}")
    return r.json()


def resolve_site_and_drive(token):
    """Devuelve (site_id, drive_id) para la biblioteca destino."""
    site = _gget(token, f"sites/{sp_site_path}")
    site_id = site['id']

    if sp_drive_name:
        drives = _gget(token, f"sites/{site_id}/drives").get('value', [])
        drive = next((d for d in drives if d.get('name') == sp_drive_name), None)
        if drive is None:
            names = [d.get('name') for d in drives]
            raise ValueError(f"Biblioteca '{sp_drive_name}' no encontrada en el site. Disponibles: {names}")
        drive_id = drive['id']
    else:
        drive_id = _gget(token, f"sites/{site_id}/drive")['id']

    return site_id, drive_id


def upload_to_sharepoint(token, site_id, drive_id, dest_relpath, content_bytes):
    """Sube un fichero a SharePoint. dest_relpath relativo a la raíz de la biblioteca.
    PUT directo para ficheros < 4 MB; nuestros CSV cuartohorarios de 2-3 meses pesan < 1 MB.

    Nota: usamos Content-Type 'application/octet-stream' (sin charset). SharePoint Graph
    rechaza algunas declaraciones de charset (p. ej. 'windows-1252'). Los bytes del cuerpo
    siguen siendo cp1252 — el header solo describe el wire format.
    """
    from urllib.parse import quote
    safe_path = "/".join(quote(seg, safe='') for seg in dest_relpath.strip('/').split('/'))

    url = f"{_GRAPH_BASE}/sites/{site_id}/drives/{drive_id}/root:/{safe_path}:/content"
    r = requests.put(
        url,
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/octet-stream'},
        data=content_bytes,
        timeout=_TIMEOUT * 2,
    )
    if not r.ok:
        raise RuntimeError(f"Graph PUT {dest_relpath} -> {r.status_code}: {r.text[:400]}")
    j = r.json()
    return {
        'id':       j.get('id'),
        'name':     j.get('name'),
        'webUrl':   j.get('webUrl'),
        'size':     j.get('size'),
        'modified': j.get('lastModifiedDateTime'),
    }

In [4]:
# ---------------------------------------------------------------------------
# CSV builder  (formato byte)
# ---------------------------------------------------------------------------
_HEADER = "Date,GHI (W/m2),T_AMB (ºC),ALB,POA (W/m2)"

_SRC_MAP = {
    'GHI (W/m2)':   'GHI_01',
    'T_AMB (ºC)': 'T_AMB_01',
    'ALB':          'ALB_avg_filt',
    'POA (W/m2)':   'POA_avg_filt',
}
_OUT_COLS = ['GHI (W/m2)', 'T_AMB (ºC)', 'ALB', 'POA (W/m2)']


def _agg_to_hourly(df):
    if df.empty:
        return df
    idx = df.set_index('Date')
    num = idx.select_dtypes(include=['number'])
    out = num.resample('1H', label='left', closed='left').mean().reset_index()
    return out


def _fmt_val(v):
    if v is None:
        return ''
    if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
        return ''
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    f = float(v)
    if f == 0.0:
        return '0'
    s = f"{f:.4f}".rstrip('0').rstrip('.')
    return s if s else '0'


def _fmt_date(ts):
    d = pd.Timestamp(ts)
    return f"{d.month}/{d.day}/{d.year} {d.hour}:{d.minute:02d}"


def build_csv(df):
    """Devuelve el contenido del CSV con line endings CRLF."""
    lines = [_HEADER]
    for _, r in df.iterrows():
        d = r['Date']
        if pd.isna(d):
            continue
        row = [_fmt_date(d)]
        for col in _OUT_COLS:
            src = _SRC_MAP.get(col)
            if src is None or src not in df.columns:
                row.append('')
            else:
                row.append(_fmt_val(r.get(src)))
        lines.append(",".join(row))
    return ("\r\n".join(lines) + "\r\n").encode('cp1252')


In [5]:
def export_plant(plant_row, token, site_id, drive_id):
    plant_code = plant_row['plant_code']

    where = [f"plant_code = '{plant_code}'"]
    if period_start:
        where.append(f"Date >= TIMESTAMP'{period_start}'")
    if period_end:
        where.append(f"Date <= TIMESTAMP'{period_end}'")
    sql = f"SELECT * FROM scada_proc WHERE {' AND '.join(where)} ORDER BY Date"
    sdf = spark.sql(sql)

    df = sdf.toPandas()
    if df.empty:
        return {'plant_code': plant_code, 'status': 'skipped',
                'reason': 'sin datos en scada_proc para el periodo'}

    df = df.sort_values('Date').reset_index(drop=True)

    if resolution == '1H':
        df = _agg_to_hourly(df)
        res_label = '1H'
    else:
        diffs = df['Date'].diff().dropna()
        res_label = f"{int(diffs.mode().iloc[0].total_seconds() // 60)}min" if not diffs.empty else 'native'

    period_start_f = pd.Timestamp(df['Date'].min()).strftime('%Y%m%d')
    period_end_f   = pd.Timestamp(df['Date'].max()).strftime('%Y%m%d')

    fname = f"{plant_code}_RawData_{period_start_f}-{period_end_f}.csv"
    content_bytes = build_csv(df)

    # 1) Audit local en OneLake (carpeta nativa del lakehouse, no shortcut)
    audit_dir = f"{LAKEHOUSE_FILES}/{local_audit_root.strip('/')}/{plant_code}"
    Path(audit_dir).mkdir(parents=True, exist_ok=True)
    audit_path = f"{audit_dir}/{fname}"
    with open(audit_path, 'wb') as f:
        f.write(content_bytes)

    # 2) Upload a SharePoint via Microsoft Graph
    dest_relpath = f"{sp_inbox_path.strip('/')}/{fname}"
    upload_info = upload_to_sharepoint(token, site_id, drive_id, dest_relpath, content_bytes)

    return {
        'plant_code':   plant_code,
        'status':       'ok',
        'rows':         len(df),
        'resolution':   res_label,
        'period_start': period_start_f,
        'period_end':   period_end_f,
        'local_audit':  audit_path.replace('/lakehouse/default/', ''),
        'sharepoint':   upload_info,
    }


# Resolver token y site/drive una sola vez por ejecución
print("Autenticando contra Microsoft Graph...")
_token  = get_graph_token()
_site_id, _drive_id = resolve_site_and_drive(_token)

results = []
for p in get_active_plants():
    plant = p['plant_code']
    print(f"\n-> {plant}")
    try:
        r = export_plant(p, _token, _site_id, _drive_id)
        if r['status'] == 'ok':
            print(f"  OK  {r['rows']:,} filas · {r['period_start']}->{r['period_end']} · {r['resolution']}")
            print(f"      audit local: {r['local_audit']}")
            # print(f"      SharePoint:  {r['sharepoint']['webUrl']}")
        else:
            print(f"  SKIPPED  {r.get('reason')}")
        results.append(r)
    except Exception as e:
        err = str(e)[:500]
        print(f"  FAILED: {err}")
        results.append({'plant_code': plant, 'status': 'failed', 'error': err})


In [6]:
import json, notebookutils
summary = {
    'plants_processed': len(results),
    'ok':       sum(1 for r in results if r['status'] == 'ok'),
    'skipped':  sum(1 for r in results if r['status'] == 'skipped'),
    'failed':   sum(1 for r in results if r['status'] == 'failed'),
    # 'uploads':  [r['sharepoint']['webUrl'] for r in results if r.get('status') == 'ok'],
}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))
